In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import Window
from snowflake.snowpark.functions import col, when, lit, count, is_integer, is_boolean, regexp_replace, length, is_array, parse_json, typeof
import pandas as pd
import re
from rapidfuzz import fuzz
import ast
from difflib import SequenceMatcher

session = get_active_session()
bronze_df = session.sql('SELECT * FROM database_name.schema_name."BRONZE_RESTAURANT"')
silver_restaurant_hours = session.sql("""
        SELECT restaurant_id, restaurant_name, operating_hours_mon, operating_hours_tue, operating_hours_wed, operating_hours_thu,
        operating_hours_fri, operating_hours_sat, operating_hours_sun
        FROM database_name.schema_name.SILVER_RESTAURANT
    """).to_pandas()

session.use_database("database_name")
session.use_schema("schema_name")


completeness_weightings = [10,10,5,5,7,7,10,10,10,4,9,3,9,9,9,9,9,
                           9,9,4,1,8,1,1,1,1,1,6,8,1,3,1,1,1,1,3,1,
                           1,1,10,5,5,5,5,5,5,1,1,1,7]

uniqueness_weightings = [10,0,0,0,0,0,10,9,9,2,10,2,0,0,0,0,0,0,0,
                         0,0,0,0,0,0,0,
                         0,3,0,0,0,0,0,0,
                         0,0,0,0,0,4,3,3,3,3,3,10,9,9,9,10, 0]
validity_weight_map = {
    # Core Identity & Location
    "restaurant_id": 10,
    "brand_name": 9,
    "country": 7,
    "subregion": 7,
    "city": 10,
    "area": 10,
    "zip_code": 10,
    "latitude": 10,
    "longitude": 10,
    
    # Contact & Digital Presence
    "telephone_number": 10,
    "email_address": 9,
    "restaurant_website": 10,
    "instagram": 6,
    "facebook": 6,
    "twitter": 6,
    
    # Attributes & Services
    "categories": 8,
    "observed_holidays": 5,
    "serves_halal": 1,
    "serves_hala": 1, # Matching original list typo
    "accepts_credit_cards": 6,
    "has_bar": 3,
    "has_delivery": 9,
    "has_outdoor_seating": 3,
    "has_smoking_area": 3,
    "has_takeout": 1,
    "has_tv": 1,
    "kid_friendly": 1,
    "offers_delivery": 9,
    "offers_pickup": 1,
    "wheelchair_accessible": 1,
    
    # Dietary Options
    "serves_dinner": 3,
    "serves_lunch": 3,
    "serves_gluten_free": 1,
    "serves_keto": 1,
    "serves_kosher": 1,
    "serves_vegan": 1,
    "serves_vegetarian": 1,
    
    # Operating Hours
    "operating_hours_mon": 10,
    "operating_hours_tue": 10,
    "operating_hours_wed": 10,
    "operating_hours_thu": 10,
    "operating_hours_fri": 10,
    "operating_hours_sat": 10,
    "operating_hours_sun": 10,
    
    # Ratings
    "food_rating": 5,
    "service_rating": 5,
    "value_rating": 5,
    "avg_platform_rating": 5,
    "avg_experience_rating": 5

}

all_column_list = ['restaurant_id', 'brand_name', 'country', 'subregion', 'city', 'area', 'address', 'latitude', 'longitude', 
'categories', 'telephone_number', 'description', 'operating_hours_mon', 'operating_hours_tue', 
'operating_hours_wed', 'operating_hours_thu', 'operating_hours_fri',
'operating_hours_sat', 'operating_hours_sun', 'accepts_credit_cards', 'has_bar', 'has_delivery',
'has_outdoor_seating', 'has_smoking_area', 'has_takeout', 'has_tv', 'kid_friendly', 'observed_holidays',
'offers_delivery', 'offers_pickup', 'serves_dinner', 'serves_gluten_free', 'serves_hala', 'serves_keto',
'serves_kosher', 'serves_lunch', 'serves_vegan', 'serves_vegetarian', 'wheelchair_accessible', 'zip_code', 
'food_rating', 'service_rating', 'value_rating', 'avg_platform_rating', 'avg_experience_rating', 'email_address',
'instagram', 'facebook', 'twitter', 'restaurant_website', 'serves_halal']



accuracy_weightings = [

  0,10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
    4, 10, 10, 10, 10, 10, 10, 10, 5, 2,
    10, 2, 2, 2, 1, 1, 8, 10, 2, 5,
    9, 9, 9, 9, 5, 9, 9, 4, 10, 7,
    7, 7, 7, 7, 9, 7, 7, 7, 10
]

accuracy_map = dict(zip(all_column_list, accuracy_weightings))
completeness_map = dict(zip(all_column_list, completeness_weightings))


In [ ]:
def completeness_for_rows(bronze_df):
    """
    Calculates completeness percentages per column and a total weighted score.
    """
    total_possible_weight = sum(completeness_map.values())

    earned_weight_expr = sum(when(col(c).is_not_null(), lit(w)).otherwise(lit(0))
                            for c,w in completeness_map.items())

    scored_df = bronze_df.with_column("ROW_COMPLETENESS", (earned_weight_expr / lit(total_possible_weight)) * 100)

    return scored_df

testing_df = completeness_for_rows(bronze_df)

row_complete = testing_df.select("RESTAURANT_ID", "BRAND_NAME", "ROW_COMPLETENESS")
# row_complete.create_or_replace_view("row_completeness")

In [ ]:
def uniqueness_row_score(bronze_df):

    """
    This function calculates a score for the uniqueness of the data by column.
    """
    
    uniqueness_weights = {
        'restaurant_id': 10,
        'address' : 10,
        'brand_name' : 7,
        'telephone_number': 10,
        'email_address': 10,
        'instagram' : 9,
        'twitter' : 9,
        'facebook' : 9,
        'restaurant_website' : 10,
        'description' : 3
    }
 
    coord_weight = 18
    total_possible_weight = sum(uniqueness_weights.values()) + coord_weight

    unique_checks = []

    for c, w in uniqueness_weights.items():
        
        window = Window.partition_by(c)
        check = when(col(c).is_null(), lit(w)) \
               .when(count('*').over(window) == 1, lit(w)) \
               .otherwise(lit(0))
        unique_checks.append(check)

    coord_window = Window.partition_by(['latitude', 'longitude'])
    
    coord_check = when((col('latitude').is_null()) | (col('longitude').is_null()), lit(18)) \
                 .when(count('*').over(coord_window) == 1, lit(18)) \
                 .otherwise(lit(0))

    unique_checks.append(coord_check)

    scored_df = bronze_df.with_column("ROW_UNIQUENESS_SCORE", (sum(unique_checks) / lit(total_possible_weight))*100)

    return scored_df

row_unique = uniqueness_row_score(bronze_df).select("RESTAURANT_ID", "BRAND_NAME", "ROW_UNIQUENESS_SCORE")
# row_unique.create_or_replace_view("row_uniqueness")

In [ ]:
manhattan_neighborhoods = [
    # Lower Manhattan
    "Alphabet City",
    "Battery Park City",
    "Bowery",
    "Chinatown",
    "Civic Center",
    "Cooperative Village",
    "Dimes Square",
    "East Village",
    "Essex Crossing",
    "Financial District",
    "Greenwich Village",
    "Hudson Square",
    "Little Australia",
    "Little Fuzhou",
    "Little Italy",
    "Loisaida",
    "Lower East Side",
    "Meatpacking District",
    "NoHo",
    "Nolita",
    "SoHo",
    "South Street Seaport",
    "South Village",
    "Tribeca",
    "Two Bridges",
    "Ukrainian Village",
    "West Village",
    "World Trade Center",

    # Midtown Manhattan
    "Columbus Circle",
    "Diamond District",
    "Flatiron District",
    "Garment District",
    "Herald Square",
    "Koreatown",
    "Madison Square",
    "Midtown South",
    "NoMad",
    "Silicon Alley",
    "Theater District",
    "Times Square",

    # West Side
    "Chelsea",
    "Hell's Kitchen",
    "Hudson Yards development",
    "Lincoln Square",
    "Manhattan Valley",
    "Penn South",
    "Pomander Walk",
    "Riverside South",
    "Upper West Side",

    # East Side
    "Carnegie Hill",
    "Gramercy Park",
    "Kips Bay",
    "Lenox Hill",
    "Murray Hill",
    "Peter Cooper Village",
    "Rose Hill",
    "Stuyvesant Square",
    "Stuyvesant Town",
    "Sutton Place",
    "Tudor City",
    "Turtle Bay",
    "Union Square",
    "Upper East Side",
    "Waterside Plaza",
    "Yorkville",

    # Upper Manhattan (above 110th St)
    "Astor Row",
    "East Harlem",
    "Fort George",
    "Hamilton Heights",
    "Harlem",
    "Hudson Heights",
    "Inwood",
    "Le Petit Sénégal",
    "Manhattanville",
    "Marble Hill",
    "Mount Morris Park",
    "Morningside Heights",
    "Sugar Hill",
    "Sylvan",
    "Washington Heights",

    # Islands
    "Ellis Island",
    "Governors Island",
    "Liberty Island",
    "Randalls and Wards Islands",
    "Roosevelt Island"
]

bronx_neighborhoods = [
    # South Bronx
    "East Morrisania",
    "Crotona Park East",
    "Hunts Point",
    "Longwood",
    "Melrose",
    "Morrisania",
    "Mott Haven",
    "Port Morris",
    "Kingsbridge Armory",

    # West Bronx
    "Bedford Park",
    "Belmont",
    "Concourse",
    "East Tremont",
    "Fordham",
    "Highbridge",
    "Jerome Park",
    "Kingsbridge",
    "Kingsbridge Heights",
    "Van Cortlandt Village",
    "Morris Heights",
    "Norwood",
    "Riverdale",
    "North Riverdale",
    "Fieldston",
    "Hudson Hill",
    "Mosholu",
    "Spuyten Duyvil",
    "Tremont",
    "University Heights",
    "West Farms",
    "Woodlawn Heights",

    # East Bronx
    "Allerton",
    "Baychester",
    "Castle Hill",
    "City Island",
    "Clason Point",
    "Harding Park",
    "Co-op City",
    "Country Club",
    "Eastchester",
    "Morris Park",
    "Olinville",
    "Pelham Bay",
    "Pelham Gardens",
    "Parkchester",
    "Pelham Parkway",
    "Soundview",
    "Throggs Neck",
    "Edgewater Park",
    "Locust Point",
    "Schuylerville",
    "Van Nest",
    "Wakefield",
    "Westchester Square",
    "Williamsbridge"
]

queens_neighborhoods = [
    "Addisleigh Park",
    "Arverne",
    "Astoria",
    "Auburndale",
    "Bayside",
    "Bay Terrace",
    "Bayswater",
    "Beechhurst",
    "Belle Harbor",
    "Bellerose",
    "Breezy Point",
    "Briarwood",
    "Broad Channel",
    "Broadway–Flushing",
    "Cambria Heights",
    "Chinatown",
    "College Point",
    "Corona",
    "Douglaston–Little Neck",
    "East Elmhurst",
    "Edgemere",
    "Elmhurst",
    "Far Rockaway",
    "Floral Park",
    "Flushing",
    "Forest Hills",
    "Fresh Meadows",
    "Fresh Pond",
    "Glendale",
    "Glen Oaks",
    "Hammels",
    "Hillside",
    "Hollis",
    "Holliswood",
    "Howard Beach",
    "Jackson Heights",
    "Jamaica",
    "Jamaica Estates",
    "Jamaica Hills",
    "Kew Gardens",
    "Kew Gardens Hills",
    "Koreatown",
    "Laurelton",
    "Locust Manor",
    "Long Island City",
    "Maspeth",
    "Meadowmere",
    "Middle Village",
    "Neponsit",
    "Ozone Park",
    "Pomonok",
    "Queensboro Hill",
    "Queensbridge",
    "Queens Village",
    "Rego Park",
    "Richmond Hill",
    "Ridgewood",
    "Rockaway",
    "Rockaway Beach",
    "Rockaway Park",
    "Rosedale",
    "Roxbury",
    "St. Albans",
    "Seaside",
    "South Jamaica",
    "South Ozone Park",
    "Springfield Gardens",
    "Sunnyside",
    "Sunnyside Gardens",
    "The Hole",
    "Whitestone",
    "Willets Point",
    "Woodhaven",
    "Woodside",
    "Wyckoff Heights"
]

staten_island_neighborhoods = [
    "Arlington",
    "Northern Castleton Corners",
    "Brighton Heights",
    "Clifton",
    "Concord",
    "Elm Park",
    "Fort Wadsworth",
    "Northern Graniteville",
    "Grymes Hill",
    "Livingston",
    "Mariners Harbor",
    "Northern Meiers Corners",
    "New Brighton",
    "Old Place",
    "Park Hill",
    "Port Ivory",
    "Port Richmond",
    "Randall Manor",
    "Rosebank",
    "Saint George",
    "Shore Acres",
    "Silver Lake",
    "Stapleton",
    "Stapleton Heights",
    "Sunnyside",
    "Tompkinsville",
    "Ward Hill",
    "West New Brighton",
    "Westerleigh",
    "Staten Island Ferry",

    # Mid-Island
    "Arrochar",
    "Bloomfield",
    "Bulls Head",
    "Southern Castleton Corners",
    "Chelsea",
    "Dongan Hills",
    "Egbertville",
    "Emerson Hill",
    "Southern Graniteville",
    "Grant City",
    "Grasmere",
    "Heartland Village",
    "Lighthouse Hill",
    "Manor Heights",
    "Southern Meiers Corners",
    "Midland Beach",
    "New Dorp",
    "New Springville",
    "Oakwood",
    "Ocean Breeze",
    "Old Town",
    "Richmondtown",
    "South Beach",
    "Teleport",
    "Todt Hill",
    "Travis",
    "Southern Willowbrook",

    # South Shore
    "Annadale",
    "Arden Heights",
    "Bay Terrace",
    "Charleston",
    "Eltingville",
    "Great Kills",
    "Greenridge",
    "Huguenot",
    "Pleasant Plains",
    "Prince's Bay",
    "Richmond Valley",
    "Rossville",
    "Sandy Ground",
    "Tottenville",
    "Woodrow"
]

brooklyn_neighborhoods = [
    "Barren Island",
    "Bath Beach",
    "Bay Ridge",
    "Bedford–Stuyvesant",
    "Bensonhurst",
    "Bergen Beach",
    "Beverley Square",
    "BoCoCa",
    "Boerum Hill",
    "Borough Park",
    "Bridge Plaza",
    "Brighton Beach",
    "Brooklyn Heights",
    "Brownsville",
    "Bushwick",
    "Canarsie",
    "Carroll Gardens",
    "Chinatown",
    "Clinton Hill",
    "Cobble Hill",
    "Columbia Street Waterfront District",
    "Coney Island",
    "Crown Heights",
    "Downtown",
    "Dumbo",
    "Dyker Heights",
    "East Flatbush",
    "East New York",
    "East Williamsburg",
    "Farragut Houses",
    "Fiske Terrace",
    "Flatbush",
    "Flatlands",
    "Fort Greene",
    "Fulton Ferry",
    "Gerritsen Beach",
    "Gowanus",
    "Gravesend",
    "Greenpoint",
    "Greenwood Heights",
    "Homecrest",
    "Kensington",
    "Little Bangladesh",
    "Little Caribbean",
    "Little Haiti",
    "Manhattan Beach",
    "Mapleton",
    "Marine Park",
    "Midwood",
    "Mill Basin",
    "Navy Yard",
    "New Utrecht",
    "Ocean Hill",
    "Ocean Parkway",
    "Pacific Park/Atlantic Yards",
    "Park Slope",
    "Park Slope Village",
    "Plumb Beach",
    "Prospect Heights",
    "Prospect Lefferts Gardens",
    "Prospect Park South",
    "Red Hook",
    "Remsen Village",
    "Sea Gate",
    "Sheepshead Bay",
    "Spring Creek",
    "South Brooklyn",
    "South Williamsburg",
    "Starrett City",
    "Sunset Park",
    "The Hole",
    "Vinegar Hill",
    "Wallabout",
    "Weeksville",
    "West Midwood",
    "White Sands",
    "Williamsburg",
    "Windsor Terrace",
    "Wingate/Pigtown",
    "Wyckoff Heights"
]
all_areas = [item for sublist in [queens_neighborhoods, brooklyn_neighborhoods, 
                 manhattan_neighborhoods, staten_island_neighborhoods, bronx_neighborhoods] 
                 for item in sublist]

In [ ]:
def validity_row_score(bronze_df):

    """
    This function calculates a score for the validity of the data by column. 
    Validity conditions are provided to ensure frameworks for the formatting, 
    range, logic, and consistency of the columns.
    """
    # Define all regular expression patterns to be used
    email_regex = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z0-9.-]+$"
    url_regex = r"^https?://[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}(/.*)?$"
    hours_pattern = r'(0?\d|1[0-2]):[0-5]\d [AaPp][Mm] - (0?\d|1[0-2]):[0-5]\d [AaPp][Mm]'

    # Define all conditions to check if the row value is valid
    validity_conditions = {
        "restaurant_id" : is_integer(col("restaurant_id")),
        "subregion" : col("subregion") == lit("NY"),
        "country" : col("country") == lit("United States"),
        "city": col("city").isin(["Manhattan", "Queens", "Staten Island", "Brooklyn"]),
        "area" : col("area").isin(all_areas),
        "zip_code" : length(regexp_replace(col("zip_code"), r'[^0-9]', '')).isin([5]),
        "latitude": col("latitude").between(-90, 90),
        "longitude": col("longitude").between(-180, 180),
        "categories": is_array(parse_json(col("categories"))),
        "telephone_number" : length(regexp_replace(col("telephone_number"),r'[^0-9]', '')).isin([10,11]),
        "observed_holidays" : is_array(parse_json(col("observed_holidays"))),
        "email_address" : (col("email_address").is_null()) | (col("email_address").regexp(email_regex)),
        "instagram" : col("instagram").contains(lit('instagram.com')),
        "facebook": (col("facebook").contains(lit('facebook.com'))) | (col("facebook").is_null()),
        "twitter" : (col("twitter").contains(lit('twitter.com'))) | (col("twitter").is_null()),
        "restaurant_website" : col("restaurant_website").regexp(url_regex),
        "serves_halal" : is_boolean(col("serves_halal")),
        "avg_experience_rating" : (typeof(col("avg_experience_rating")) == lit('DOUBLE')) & (length(regexp_replace(col("avg_platform_rating"),r'[^0-9]', '')).isin([4])),
        "avg_platform_rating" : (typeof(col("avg_platform_rating")) == lit('DOUBLE')) & (length(regexp_replace(col("avg_platform_rating"),r'[^0-9]', '')).isin([4])),
        "food_rating" : (typeof(col("food_rating")) == lit('DOUBLE')) & (length(regexp_replace(col("food_rating"),r'[^0-9]', '')).isin([4])),
        "value_rating" : (typeof(col("value_rating")) == lit('DOUBLE')) & (length(regexp_replace(col("value_rating"),r'[^0-9]', '')).isin([4])),
        "service_rating" : (typeof(col("service_rating")) == lit('DOUBLE')) & (length(regexp_replace(col("service_rating"),r'[^0-9]', '')).isin([4]))
        
    }

    bool_columns = ["accepts_credit_cards","has_bar", "has_delivery","has_outdoor_seating",
        "has_smoking_area", "has_takeout","has_tv","kid_friendly","offers_delivery",
        "offers_pickup","serves_dinner","serves_gluten_free", "serves_hala", "serves_keto",
        "serves_kosher", "serves_lunch", "serves_vegan", "serves_vegetarian", "wheelchair_accessible"]

    for item in bool_columns:
        validity_conditions[item] = is_boolean(col(item))
        
    multi_shift = f"^{hours_pattern}(,\\s{hours_pattern})?$"
    for day in ["mon", "tue", "wed", "thu", "fri", "sat", "sun"]:
        validity_conditions[f"operating_hours_{day}"] = col(f"operating_hours_{day}").regexp(multi_shift)

    total_possible_weight = sum(validity_weight_map.values())
    valid_check = []

    for column, condition in validity_conditions.items():
        
        check = when(col(column).is_null() | condition, lit(validity_weight_map[column])) \
               .otherwise(lit(0))
        valid_check.append(check)
    
    scored_df = bronze_df.with_column("ROW_VALIDITY_SCORE", (sum(valid_check)/lit(total_possible_weight))*100) 

    return scored_df

row_valid = validity_row_score(bronze_df).select("RESTAURANT_ID", "BRAND_NAME", "ROW_VALIDITY_SCORE")
# row_valid.create_or_replace_view("row_validity")

In [ ]:
def explode_operating_hours(hours_str):
    """
    The operating hours from the API come in a sinlge string. This function explodes each record
    into 7 separate records that can easily be compared using fuzzy matching to the hours in 
    our bronze restaurant table.
    """
    # Define the day prefixes that are shown in the API results
    days_map = {'Mo': 'mon', 'Tu': 'tue', 'We': 'wed', 'Th': 'thu', 'Fr': 'fri', 'Sa': 'sat', 'Su': 'sun'}
    days_list = list(days_map.keys())
    exploded = {f"operating_hours_{suffix}": None for suffix in days_map.values()}
    
    if not hours_str or str(hours_str).strip() == "" or hours_str == 'None':
        return exploded

    # Replace || and ; with a comma to create a clean list
    # but only if the comma is followed by a Day code
    raw_str = str(hours_str).replace('||', ';').replace(';', ',')
    
    # Split by comma ONLY when followed by a day name
    sections = re.split(r',(?=\s*(?:Mo|Tu|We|Th|Fr|Sa|Su))', raw_str)
    
    for section in sections:
        section = section.strip()
        # Separate Day Part from Time Part
        match = re.match(r'^([A-Z][a-z](?:-[A-Z][a-z])?)\s*(.*)$', section)
        if match:
            day_part, time_part = match.groups()
            
            # Handle ranges like Mo-Th
            if '-' in day_part:
                try:
                    start_day, end_day = day_part.split('-')
                    start_idx = days_list.index(start_day)
                    end_idx = days_list.index(end_day)
                    # If range is Su-Tu (wraps around), handle that or just slice
                    if start_idx <= end_idx:
                        target_days = days_list[start_idx:end_idx+1]
                    else: # Wraps around weekend
                        target_days = days_list[start_idx:] + days_list[:end_idx+1]
                except:
                    target_days = [day_part]
            else:
                target_days = [day_part]

            for d in target_days:
                if d in days_map:
                    exploded[f"operating_hours_{days_map[d]}"] = time_part
        else:
            # Fallback for strings that are JUST times (applies to all days)
            if re.search(r'\d', section): 
                for key in exploded: exploded[key] = section

    return exploded

In [ ]:
bronze_rest = session.sql("""
    SELECT ADDRESS, BRAND_NAME, SUBREGION, ZIP_CODE, AREA, CITY,
    TELEPHONE_NUMBER, RESTAURANT_WEBSITE, WHEELCHAIR_ACCESSIBLE, COUNTRY
    FROM database_name.schema_name.BRONZE_RESTAURANT
""")

bronze_rest = bronze_rest.with_column('CITY', when((col('CITY') == lit('New York')) | (col('CITY') == lit('New York City')), 'Manhattan').otherwise(col('CITY')))
bronze_rest = bronze_rest.to_pandas()

API_results = session.table("database_name.schema_name.RESTAURANT_API_RESULTS").to_pandas()

cols_bronze_to_API = {
    "COUNTRY":"COUNTRY",
    "SUBREGION": "CITY", 
    "ZIP_CODE": "ZIP_CODE", 
    "RESTAURANT_WEBSITE": "WEBSITE",
    "BRAND_NAME": "RESTAURANT_NAME",
    "WHEELCHAIR_ACCESSIBLE": "WHEELCHAIR",
    "ADDRESS":"ADDRESS",
    "TELEPHONE_NUMBER": "PHONE"}

accuracy_map_bronze = pd.DataFrame(index=bronze_rest.index)
threshold = 90

# main columns
for left_col, right_col in cols_bronze_to_API.items():
    
    map_col_name = f"{left_col}_vs_{right_col}_accuracy"
   
    if left_col == 'BRAND_NAME':
        results = []
        for b_val, a_val in zip(bronze_rest[left_col], API_results[right_col]):
            if a_val.strip().lower() in ["", "none", "nan", "None"]:
                results.append(None)
            elif b_val in a_val or a_val in b_val:
                results.append(1)
            else:
                ratio = fuzz.ratio(str(b_val).strip().lower(), str(a_val).strip().lower())
                results.append(1 if ratio >= threshold else 0)
                
        accuracy_map_bronze[map_col_name] = results
        continue

# wheelchair
    map_col_name = f"{left_col}_vs_{right_col}_accuracy"
    if left_col == 'WHEELCHAIR_ACCESSIBLE':
        results = []
        for b_val, a_val in zip(bronze_rest[left_col], API_results[right_col]):
            if a_val.strip().lower() in ["", "none", "nan", "None"]:
                results.append(None)
            elif a_val.strip().lower() in ["yes", "limited"] and b_val == True:
                results.append(1)
            else:
                ratio = fuzz.ratio(str(b_val).strip().lower(), str(a_val).strip().lower())
                results.append(1 if ratio >= threshold else 0)
    
        accuracy_map_bronze[map_col_name] = results
        continue

# address
    if left_col == 'ADDRESS':

        results = []
        street_endings = ['broadway', 'blvd','boulevard', 'ave', 'avenue', 'st', 'street', 'pl', 'place']
        
        street_mappings = {'broadway': 'Broadway',
                           'boulevard' : 'Boulevard',
                          'blvd':'Boulevard', 
                          'ave': 'Avenue',
                          'avenue':'Avenue',
                          'street':'Street',
                          'st': 'Street', 
                          'pl': 'Place',
                          'place':'Place'}
        
        cities = bronze_rest["CITY"]
        areas = bronze_rest["AREA"]
        
        found = False
    
        for i, (b_val, a_val) in enumerate(zip(bronze_rest[left_col], API_results[right_col])):
    
            if a_val.strip().lower() in ["", "none", "nan", "None"]:
                results.append(None)
                continue
    
            for ending in street_endings:
                if ending in b_val.lower().strip():
                    
                    new_address = b_val.lower().rpartition(ending)
                    
                    address_area = str(new_address[0].title() + street_mappings[new_address[1]].title() + ', ' + (areas.iloc[i]).title())
                    address_area.title()
                    address_city = str(new_address[0].title() + street_mappings[new_address[1]].title() + ', ' + (cities.iloc[i]).title())
                
                    found = True
                    break
                    
            if found:
                
                ratio1 = fuzz.ratio(address_area, a_val)
                ratio2 = fuzz.ratio(address_city, a_val)
                
                results.append(1 if (ratio1 >= threshold) or (ratio2 >= threshold) else 0)
            else:
                ratio = fuzz.ratio(b_val, a_val)
                results.append(1 if ratio >= threshold else 0)
    
        accuracy_map_bronze[map_col_name] = results
        continue
    
# telephone number
    if left_col == 'TELEPHONE_NUMBER':
        results = []
        for b_val, a_val in zip(bronze_rest[left_col], API_results[right_col]):
            b_str = str(b_val)
            a_str = str(a_val).strip().lower()
            clean_a_str = re.sub(r'[^0-9+]', '', a_str)
            clean_b_str = re.sub(r'[^0-9+]', '', b_str)
            
            plus_count = b_str.count('+')
            
            if clean_a_str in ["", "none", "nan", "None"]:
                results.append(None)
                continue
            match_found = 0
            if plus_count > 1:
                s = 0
                for i in range(plus_count):
                    
                    current_index = clean_b_str.index('+', s)
                    current_number = clean_b_str[current_index: current_index+11]
                    s = current_index + 1
                    
                    ratio = fuzz.ratio(str(current_number).strip().lower(), str(clean_a_str).strip().lower())
                    if ratio >= threshold or clean_a_str in current_number:
                        match_found = 1/plus_count
                        break
                     
                results.append(match_found)
                
            elif plus_count == 1:
                ratio = fuzz.ratio(clean_b_str.strip().lower(), str(clean_a_str).strip().lower())
                results.append(1 if ratio >= threshold else 0)
            else:
                ratio = fuzz.ratio(clean_b_str.strip().lower(), str(clean_a_str).strip().lower())
                results.append(1 if ratio >= threshold else 0)
    
        accuracy_map_bronze[map_col_name] = results
        continue
        
    results = []
    for b_val, a_val in zip(bronze_rest[left_col], API_results[right_col]):
        if a_val.strip().lower() in ["", "none", "nan", "None"]:
            results.append(None)
        else:
            ratio = fuzz.ratio(str(b_val).strip().lower(), str(a_val).strip().lower())
            results.append(1 if ratio >= threshold else 0)
    
    accuracy_map_bronze[map_col_name] = results

bronze_acc_map = accuracy_map_bronze

In [ ]:
def clean_hours(s):
    """
    This small function cleans the operating hours before fuzzy matching.
    It ensures that we correctly identify 'Closed' in the hours and keep
    comparisons clean with lower case strings and no extra spacing.
    """
    
    if pd.isna(s) or s == "Closed": return "closed"
    # Remove all spaces and lowercase the string
    return str(s).replace(" ", "").lower()

In [ ]:
def accuracy_score(accuracy_map):

    """
    This function combines the bronze_acc_map and the accuracy values for the
    7 daily operating hours. We combine the pandas dataframe and the snowpark dataframes
    all on restaurant ids.
    """
    
    silver_restaurant_hours.columns = [c.lower() for c in silver_restaurant_hours.columns]

    API_results = session.table("database_name.schema_name.RESTAURANT_API_RESULTS")
    
    collected_api_hours = [row[0] for row in API_results.select('HOURS').collect()]
    
    # initialize lists for daily operating hours
    monday = []
    tuesday = []
    wednesday = []
    thursday = []
    friday = []
    saturday = []
    sunday = []

    for hour_set in collected_api_hours:

        res = explode_operating_hours(hour_set)
        
        monday.append(res.get('operating_hours_mon', 'Closed'))
        tuesday.append(res.get('operating_hours_tue', 'Closed'))
        wednesday.append(res.get('operating_hours_wed', 'Closed'))
        thursday.append(res.get('operating_hours_thu', 'Closed'))
        friday.append(res.get('operating_hours_fri', 'Closed'))
        saturday.append(res.get('operating_hours_sat', 'Closed'))
        sunday.append(res.get('operating_hours_sun', 'Closed'))

    # put all daily lists into a dictionary to transform into a data frame for API hours
    hours_check_dict = {}
    hours_check_dict['operating_hours_mon2'] = monday
    hours_check_dict['operating_hours_tue2'] = tuesday
    hours_check_dict['operating_hours_wed2'] = wednesday
    hours_check_dict['operating_hours_thu2'] = thursday
    hours_check_dict['operating_hours_fri2'] = friday
    hours_check_dict['operating_hours_sat2'] = saturday
    hours_check_dict['operating_hours_sun2'] = sunday

    api_pd = pd.DataFrame(hours_check_dict)

    days = ['mon', 'tue', 'wed', 'thu', 'fri', 'sat', 'sun']
    accuracy_results = {f'operating_hours_{day}': [] for day in days}

    # tterate through all the rows (351 rows)
    for i in range(len(silver_restaurant_hours)):
        
        for day in days:

            val1 = silver_restaurant_hours.iloc[i][f'operating_hours_{day}']
            val2 = api_pd.iloc[i][f'operating_hours_{day}2']
            
            if pd.isna(val2) or str(val2).strip().lower() in ['none', '', 'nan', 'null']:
                accuracy_results[f'operating_hours_{day}'].append(None)
                continue
                
            # Clean strings
            c1, c2 = clean_hours(val1), clean_hours(val2)
    
            if c1 == c2:
                score = 1
            else:
                similarity = SequenceMatcher(None, c1, c2, autojunk=True).ratio()
                score = 1 if similarity >= 0.80 else 0
            
            # Append to the correct list in our dictionary
            accuracy_results[f'operating_hours_{day}'].append(score)

    accuracy_df_days = pd.DataFrame(accuracy_results)
    accuracy_df = pd.concat([bronze_acc_map, accuracy_df_days], axis = 1)

    weights = [
    accuracy_map['country'],                  
    accuracy_map['subregion'],                
    accuracy_map['zip_code'],                 
    accuracy_map['restaurant_website'],       
    accuracy_map['brand_name'],               
    accuracy_map['wheelchair_accessible'],    
    accuracy_map['address'],                  
    accuracy_map['telephone_number'],         
    accuracy_map['operating_hours_mon'],      
    accuracy_map['operating_hours_tue'],     
    accuracy_map['operating_hours_wed'],      
    accuracy_map['operating_hours_thu'],      
    accuracy_map['operating_hours_fri'],      
    accuracy_map['operating_hours_sat'],      
    accuracy_map['operating_hours_sun']       
]
    weighted_values = accuracy_df.mul(weights, axis = 1)
    row_weighted_sum = weighted_values.sum(axis=1)

    # create the mask to only multiply non n/a values
    mask = accuracy_df.notna().astype(int)
    row_total_possible_weight = mask.mul(weights, axis = 1).sum(axis=1)

    
    accuracy_df['ROW_ACCURACY'] = (row_weighted_sum / row_total_possible_weight) * 100
    restaurant_id = [row['RESTAURANT_ID'] for row in bronze_df.collect()]
    accuracy_df['RESTAURANT_ID'] = restaurant_id
    
    return accuracy_df
    
row_accurate = session.write_pandas(df = accuracy_score(accuracy_map), table_name= 'ROW_ACCURACY', auto_create_table=True, overwrite = True)

all_dim_df = row_complete.join(row_unique, on=["RESTAURANT_ID", "BRAND_NAME"]) \
              .join(row_valid, on=["RESTAURANT_ID", "BRAND_NAME"]) \
                .join(row_accurate, on=["RESTAURANT_ID"]) \
              .select(
                  row_complete["RESTAURANT_ID"],
                  row_complete["BRAND_NAME"],
                  row_complete["ROW_COMPLETENESS"].alias("COMPLETENESS"),
                  row_unique["ROW_UNIQUENESS_SCORE"].alias("UNIQUENESS"),
                  row_valid["ROW_VALIDITY_SCORE"].alias("VALIDITY"),
                  row_accurate["ROW_ACCURACY"].alias("ACCURACY")
              )

all_dim_df.write.mode('overwrite').save_as_table("database_name.schema_name.all_dim_df")

In [ ]:
categories = session.sql("""
SELECT RESTAURANT_ID, RESTAURANT_NAME, CATEGORIES FROM database_name.schema_name.SILVER_RESTAURANT""").to_pandas()

categories['CATEGORIES'] = categories['CATEGORIES'].apply(ast.literal_eval)
categories = categories.reset_index(drop = True)
categories_expanded = categories.explode('CATEGORIES')
categories_expanded = session.create_dataframe(categories_expanded)

categories_expanded.write.mode('overwrite').save_as_table('database_name.schema_name.RESTAURANT_CATEGORIES_FILTERING')